In [ ]:
#@title Install

!git clone https://github.com/ultralytics/yolov5.git
!pip install -r /content/yolov5/requirements.txt
!pip install pytesseract

!pip install  opencv-python pillow openpyxl torch torchvision tqdm
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-python==4.10.0.84

In [ ]:
#@title Import

from ultralytics import YOLO
from IPython.display import display
from IPython.display import clear_output
from PIL import Image
from google.colab import files
from google.colab.patches import cv2_imshow

import os
import numpy as np
import glob
import time
import pytesseract

import cv2
import shutil
import torch

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Đảm bảo sử dụng đúng mô hình YOLOv5 được tải từ PyTorch Hub
model_v5 = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/best.pt', force_reload=False)

lang = 'eng'
config = r'--oem 3 --psm 6'

# Kiểm tra mô hình đã được tải thành công
print("Đã tải thành công mô hình YOLOv5!")


In [ ]:
#@title Expand & Crop

def edge_lengths(points):
    tl = np.array(points["top-left"], dtype=np.float32)
    tr = np.array(points["top-right"], dtype=np.float32)
    br = np.array(points["bot-right"], dtype=np.float32)
    bl = np.array(points["bot-left"], dtype=np.float32)

    top = np.linalg.norm(tr - tl)
    right = np.linalg.norm(br - tr)
    bottom = np.linalg.norm(bl - br)
    left = np.linalg.norm(tl - bl)

    return top, right, bottom, left

def quad_area(points):
    pts = np.float32([
        points["top-left"],
        points["top-right"],
        points["bot-right"],
        points["bot-left"]
    ])

    x = pts[:, 0]
    y = pts[:, 1]

    return 0.5 * abs(
        np.dot(x, np.roll(y, -1))
        - np.dot(y, np.roll(x, -1))
    )

def debug_points(image, points):
    """
    Vẽ 4 điểm TL/TR/BR/BL và nối thành quadrilateral.
    """

    debug = image.copy()

    colors = {
        "top-left":  (255, 0, 0),      # Blue
        "top-right": (0, 255, 0),      # Green
        "bot-right": (0, 0, 255),      # Red
        "bot-left":  (255, 255, 0)     # Cyan
    }

    # Vẽ điểm + tên
    for name, p in points.items():

        x, y = map(int, p)

        cv2.circle(
            debug,
            (x, y),
            15,
            colors.get(name, (255, 255, 255)),
            -1
        )

        cv2.putText(
            debug,
            name,
            (x + 15, y - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            colors.get(name, (255, 255, 255)),
            2,
            cv2.LINE_AA
        )

        # Hiện tọa độ
        cv2.putText(
            debug,
            f"({x},{y})",
            (x + 15, y + 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

    # Thứ tự chuẩn
    ordered = [
        points["top-left"],
        points["top-right"],
        points["bot-right"],
        points["bot-left"]
    ]

    # Nối 4 cạnh
    for i in range(4):

        p1 = tuple(map(int, ordered[i]))
        p2 = tuple(map(int, ordered[(i + 1) % 4]))

        cv2.line(
            debug,
            p1,
            p2,
            (0, 255, 255),
            4,
            cv2.LINE_AA
        )

    return debug

def expand_card_points(points,
                       top_margin=0.02,
                       bottom_margin=0.08,
                       left_margin=0.01,
                       right_margin=0.01):

    tl = np.array(points["top-left"], dtype=np.float32)
    tr = np.array(points["top-right"], dtype=np.float32)
    br = np.array(points["bot-right"], dtype=np.float32)
    bl = np.array(points["bot-left"], dtype=np.float32)

    # Vector theo 2 cạnh dọc của card
    left_vec = bl - tl
    right_vec = br - tr

    left_len = np.linalg.norm(left_vec)
    right_len = np.linalg.norm(right_vec)

    if left_len > 0:
        left_unit = left_vec / left_len
    else:
        left_unit = np.array([0, 1], dtype=np.float32)

    if right_len > 0:
        right_unit = right_vec / right_len
    else:
        right_unit = np.array([0, 1], dtype=np.float32)

    # Nới phía trên
    tl = tl - left_unit * (left_len * top_margin)
    tr = tr - right_unit * (right_len * top_margin)

    # Nới phía dưới
    bl = bl + left_unit * (left_len * bottom_margin)
    br = br + right_unit * (right_len * bottom_margin)

    # Nới trái/phải theo vector ngang
    top_vec = tr - tl
    bottom_vec = br - bl

    top_len = np.linalg.norm(top_vec)
    bottom_len = np.linalg.norm(bottom_vec)

    if top_len > 0:
        top_unit = top_vec / top_len
    else:
        top_unit = np.array([1, 0], dtype=np.float32)

    if bottom_len > 0:
        bottom_unit = bottom_vec / bottom_len
    else:
        bottom_unit = np.array([1, 0], dtype=np.float32)

    tl = tl - top_unit * (top_len * left_margin)
    bl = bl - bottom_unit * (bottom_len * left_margin)

    tr = tr + top_unit * (top_len * right_margin)
    br = br + bottom_unit * (bottom_len * right_margin)

    return {
        "top-left": tl,
        "top-right": tr,
        "bot-right": br,
        "bot-left": bl
    }

def perspective_crop(image, points, scale=1.0,
                     top_margin=0.02,
                     bottom_margin=0.08):

    points = expand_card_points(
        points,
        top_margin=top_margin,
        bottom_margin=bottom_margin,
        left_margin=0.01,
        right_margin=0.01
    )

    tl = np.array(points["top-left"], dtype=np.float32)
    tr = np.array(points["top-right"], dtype=np.float32)
    br = np.array(points["bot-right"], dtype=np.float32)
    bl = np.array(points["bot-left"], dtype=np.float32)

    width_top = np.linalg.norm(tr - tl)
    width_bottom = np.linalg.norm(br - bl)

    height_left = np.linalg.norm(bl - tl)
    height_right = np.linalg.norm(br - tr)

    output_width = int(max(width_top, width_bottom) * scale)
    output_height = int(max(height_left, height_right) * scale)

    src = np.float32([tl, tr, br, bl])

    dst = np.float32([
        [0, 0],
        [output_width - 1, 0],
        [output_width - 1, output_height - 1],
        [0, output_height - 1]
    ])

    M = cv2.getPerspectiveTransform(src, dst)

    cropped = cv2.warpPerspective(
        image,
        M,
        (output_width, output_height),
        flags=cv2.INTER_LANCZOS4,
        borderMode=cv2.BORDER_REPLICATE
    )

    return cropped

In [ ]:
#@title Auto Rotate
# import pytesseract

# def rotate_crop(image, angle):
#     if angle == 0:
#         return image.copy()

#     elif angle == 90:
#         return cv2.rotate(
#             image,
#             cv2.ROTATE_90_CLOCKWISE
#         )

#     elif angle == 180:
#         return cv2.rotate(
#             image,
#             cv2.ROTATE_180
#         )

#     elif angle == 270:
#         return cv2.rotate(
#             image,
#             cv2.ROTATE_90_COUNTERCLOCKWISE
#         )

#     else:
#         raise ValueError(
#             "angle phải là 0, 90, 180 hoặc 270"
#         )

# def auto_orient_crop(
#     crop,
#     debug=True
# ):
#     """
#     Đưa CCCD về orientation ngang.

#     Hiện tại dùng aspect ratio để xử lý
#     orientation 90/270.

#     Returns:
#         oriented_crop
#         angle
#     """

#     h, w = crop.shape[:2]

#     if debug:
#         print("\n=== AUTO ORIENT CROP ===")
#         print(
#             f"Input crop: {w} x {h}"
#         )

#     # CCCD ID-1 có dạng ngang
#     if h > w:

#         # Crop đang đứng dọc.
#         # Thử xoay 90° trước.
#         oriented = rotate_crop(
#             crop,
#             90
#         )

#         angle = 90

#         if debug:
#             print(
#                 "Crop dọc → rotate 90°"
#             )

#     else:

#         oriented = crop.copy()
#         angle = 0

#         if debug:
#             print(
#                 "Crop đã ngang → giữ nguyên"
#             )

#     if debug:
#         print(
#             "Output:",
#             oriented.shape
#         )

#     return oriented, angle

# def orientation_score(detections, image_shape):
#     """
#     Chấm điểm xem các class corner có nằm đúng
#     vị trí tương đối hay không.

#     Không yêu cầu đủ 4 góc.
#     """

#     h, w = image_shape[:2]

#     score = 0.0

#     for name, info in detections.items():

#         if name not in CORNER_NAMES:
#             continue

#         x1, y1, x2, y2 = map(
#             float,
#             info["box"]
#         )

#         cx = (x1 + x2) / 2
#         cy = (y1 + y2) / 2

#         nx = cx / w
#         ny = cy / h

#         conf = float(info["conf"])

#         # ---------------------------------------------
#         # Vị trí kỳ vọng
#         # ---------------------------------------------

#         if name == "top-left":

#             # x nhỏ, y nhỏ
#             position_score = (
#                 (1 - nx) +
#                 (1 - ny)
#             ) / 2

#         elif name == "top-right":

#             # x lớn, y nhỏ
#             position_score = (
#                 nx +
#                 (1 - ny)
#             ) / 2

#         elif name == "bot-right":

#             # x lớn, y lớn
#             position_score = (
#                 nx +
#                 ny
#             ) / 2

#         elif name == "bot-left":

#             # x nhỏ, y lớn
#             position_score = (
#                 (1 - nx) +
#                 ny
#             ) / 2

#         score += position_score * conf

#     return score

# def rotation_score(num_points, quality, confidence):
#     return (
#         num_points * 100
#         + quality * 20
#         + confidence * 5
#     )


# def auto_rotate_card(model, image, conf_threshold=0.20, debug=False):

#     candidates = []

#     rotations = [0, 90, 180, 270]

#     for angle in rotations:

#         # =================================================
#         # Rotate
#         # =================================================

#         if angle == 0:

#             rotated = image.copy()

#         elif angle == 90:

#             rotated = cv2.rotate(
#                 image,
#                 cv2.ROTATE_90_CLOCKWISE
#             )

#         elif angle == 180:

#             rotated = cv2.rotate(
#                 image,
#                 cv2.ROTATE_180
#             )

#         else:

#             rotated = cv2.rotate(
#                 image,
#                 cv2.ROTATE_90_COUNTERCLOCKWISE
#             )

#         # =================================================
#         # YOLO
#         # =================================================

#         detections = get_detections(
#             model,
#             rotated,
#             conf_threshold
#         )

#         n = len(detections)

#         if n < 2:
#             continue

#         h, w = rotated.shape[:2]

#         # =================================================
#         # 1. Position score
#         # =================================================

#         position_score = orientation_score(
#             detections,
#             rotated.shape
#         )

#         # =================================================
#         # 2. Confidence score
#         # =================================================

#         confidence_score = sum(
#             d["conf"]
#             for d in detections.values()
#         )

#         # =================================================
#         # 3. Geometry score
#         # =================================================

#         geometry_score = 0.0

#         if n == 4:

#             pts = {}

#             for name, data in detections.items():

#                 x1, y1, x2, y2 = data["box"]

#                 pts[name] = np.array([
#                     (x1 + x2) / 2,
#                     (y1 + y2) / 2
#                 ], dtype=np.float32)

#             try:

#                 tl = pts["top-left"]
#                 tr = pts["top-right"]
#                 br = pts["bot-right"]
#                 bl = pts["bot-left"]

#                 width_top = np.linalg.norm(tr - tl)
#                 width_bottom = np.linalg.norm(br - bl)

#                 height_left = np.linalg.norm(bl - tl)
#                 height_right = np.linalg.norm(br - tr)

#                 if (
#                     width_top > 10 and
#                     width_bottom > 10 and
#                     height_left > 10 and
#                     height_right > 10
#                 ):

#                     width = (
#                         width_top +
#                         width_bottom
#                     ) / 2

#                     height = (
#                         height_left +
#                         height_right
#                     ) / 2

#                     ratio = width / height

#                     # CCCD nằm ngang sau normalize
#                     # tỷ lệ khoảng 1.5 - 1.7
#                     ratio_error = abs(
#                         ratio - 1.586
#                     )

#                     ratio_score = max(
#                         0,
#                         1.0 - ratio_error / 1.0
#                     )

#                     # Kiểm tra hai cạnh trên/dưới
#                     top_vec = tr - tl
#                     bottom_vec = br - bl

#                     # Kiểm tra hai cạnh trái/phải
#                     left_vec = bl - tl
#                     right_vec = br - tr

#                     def parallel_score(v1, v2):

#                         n1 = np.linalg.norm(v1)
#                         n2 = np.linalg.norm(v2)

#                         if n1 == 0 or n2 == 0:
#                             return 0

#                         cos_angle = abs(
#                             np.dot(v1, v2) /
#                             (n1 * n2)
#                         )

#                         return cos_angle

#                     horizontal_score = parallel_score(
#                         top_vec,
#                         bottom_vec
#                     )

#                     vertical_score = parallel_score(
#                         left_vec,
#                         right_vec
#                     )

#                     geometry_score = (
#                         ratio_score * 0.4 +
#                         horizontal_score * 0.3 +
#                         vertical_score * 0.3
#                     )

#             except Exception:
#                 geometry_score = 0.0

#         # =================================================
#         # 4. Final score
#         # =================================================

#         # Không để số lượng điểm áp đảo hoàn toàn
#         final_score = (
#             n * 2.0 +
#             position_score * 2.0 +
#             confidence_score * 0.5 +
#             geometry_score * 2.0
#         )

#         candidates.append({
#             "angle": angle,
#             "image": rotated,
#             "detections": detections,
#             "num_points": n,
#             "position_score": position_score,
#             "confidence_score": confidence_score,
#             "geometry_score": geometry_score,
#             "score": final_score
#         })

#         print(
#             f"Rotation {angle}: "
#             f"{n}/4 | "
#             f"position={position_score:.3f} | "
#             f"conf={confidence_score:.3f} | "
#             f"geometry={geometry_score:.3f} | "
#             f"FINAL={final_score:.3f}"
#         )

#     # =====================================================
#     # Không có candidate
#     # =====================================================

#     if not candidates:

#         raise ValueError(
#             "Không tìm được ít nhất 2 góc "
#             "ở bất kỳ rotation nào."
#         )

#     # =====================================================
#     # Chọn best
#     # =====================================================

#     best = max(
#         candidates,
#         key=lambda x: x["score"]
#     )

#     print(
#         f"\nSelected rotation: "
#         f"{best['angle']}°"
#     )

#     print(
#         f"Detected: "
#         f"{best['num_points']}/4"
#     )

#     # =====================================================
#     # DEBUG: hiển thị 4 rotation
#     # =====================================================

#     if debug:

#         import matplotlib.pyplot as plt

#         fig, axes = plt.subplots(
#             1,
#             4,
#             figsize=(20, 6)
#         )

#         for ax, angle in zip(
#             axes,
#             rotations
#         ):

#             candidate = next(
#                 (
#                     c for c in candidates
#                     if c["angle"] == angle
#                 ),
#                 None
#             )

#             if candidate is None:

#                 ax.set_title(
#                     f"{angle}°\nNo detection"
#                 )

#                 ax.axis("off")
#                 continue

#             ax.imshow(
#                 cv2.cvtColor(
#                     candidate["image"],
#                     cv2.COLOR_BGR2RGB
#                 )
#             )

#             ax.set_title(
#                 f"{angle}° | "
#                 f"{candidate['num_points']}/4\n"
#                 f"score={candidate['score']:.2f}"
#             )

#             ax.axis("off")

#         plt.tight_layout()
#         plt.show()

#     return (
#         best["image"],
#         best["detections"],
#         best["angle"]
#     )

In [ ]:
#@title Missing Corners Detection

def get_detections(model, image, conf_threshold=0.25):
    """
    Chạy YOLOv5 và trả về detections dạng:

    {
        'top-right': {
            'conf': 0.76,
            'box': [x1, y1, x2, y2]
        },
        ...
    }
    """

    # YOLO inference
    results = model(image)

    detections = {}

    # results.xyxy[0]:
    # [x1, y1, x2, y2, confidence, class_id]
    for det in results.xyxy[0].cpu().numpy():

        x1, y1, x2, y2, conf, cls = det

        if conf < conf_threshold:
            continue

        class_name = model.names[int(cls)]

        # Nếu một class xuất hiện nhiều lần,
        # chỉ giữ detection có confidence cao nhất
        if (
            class_name not in detections
            or conf > detections[class_name]["conf"]
        ):
            detections[class_name] = {
                "conf": float(conf),
                "box": [
                    float(x1),
                    float(y1),
                    float(x2),
                    float(y2)
                ]
            }

    return detections


CORNER_NAMES = [
    "top-left",
    "top-right",
    "bot-right",
    "bot-left"
]

import cv2
import numpy as np


def complete_right_corners(image, points):
    """
    YOLO chỉ detect được:
        top-right
        bot-right

    Trường hợp card chạm mép trái ảnh.
    Tìm cạnh trên + cạnh dưới rồi kéo về x = 0.
    """

    h, w = image.shape[:2]

    tr = np.array(points["top-right"], dtype=np.float32)
    br = np.array(points["bot-right"], dtype=np.float32)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Làm nhẹ ảnh để giảm nhiễu
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(gray, 50, 150)

    # =====================================================
    # Tìm các đường gần ngang
    # =====================================================

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=100,
        minLineLength=int(w * 0.25),
        maxLineGap=80
    )

    top_candidates = []
    bottom_candidates = []

    if lines is not None:
        lines = np.asarray(lines).reshape(-1, 4)
        for line in lines:

            x1, y1, x2, y2 = map(float, line)

            dx = x2 - x1
            dy = y2 - y1

            if abs(dx) < 50:
                continue

            angle = np.degrees(np.arctan2(dy, dx))

            # Chỉ lấy đường tương đối ngang
            if abs(angle) > 12:
                continue

            # Chuẩn hóa hướng trái -> phải
            if x1 > x2:
                x1, x2 = x2, x1
                y1, y2 = y2, y1

            # Y trung bình của line
            ym = (y1 + y2) / 2

            # -------------------------------------------------
            # Top edge
            # -------------------------------------------------
            # Phải nằm gần vùng phía trên của TR
            if abs(ym - tr[1]) < h * 0.18:

                # Ưu tiên line chạy qua gần TR
                if min(x1, x2) < tr[0] < max(x1, x2) + 100:

                    top_candidates.append(
                        (x1, y1, x2, y2)
                    )

            # -------------------------------------------------
            # Bottom edge
            # -------------------------------------------------
            if abs(ym - br[1]) < h * 0.18:

                if min(x1, x2) < br[0] < max(x1, x2) + 100:

                    bottom_candidates.append(
                        (x1, y1, x2, y2)
                    )

    # =====================================================
    # Hàm chọn line tốt nhất
    # =====================================================

    def choose_line(candidates, target):

        if not candidates:
            return None

        best = None
        best_score = float("inf")

        for line in candidates:

            x1, y1, x2, y2 = line

            # y dự đoán tại vị trí target.x
            if abs(x2 - x1) < 1:
                continue

            slope = (y2 - y1) / (x2 - x1)

            y_at_target = y1 + slope * (target[0] - x1)

            distance = abs(y_at_target - target[1])

            # Ưu tiên:
            # - gần điểm YOLO
            # - line dài
            length = np.hypot(x2 - x1, y2 - y1)

            score = distance - 0.01 * length

            if score < best_score:
                best_score = score
                best = line

        return best

    top_line = choose_line(top_candidates, tr)
    bottom_line = choose_line(bottom_candidates, br)

    # =====================================================
    # Nếu tìm được cả 2 cạnh
    # =====================================================

    if top_line is not None and bottom_line is not None:

        def extrapolate_to_x0(line):
            x1, y1, x2, y2 = line

            slope = (y2 - y1) / (x2 - x1)

            y0 = y1 + slope * (0 - x1)

            return np.array([0, y0], dtype=np.float32)

        tl = extrapolate_to_x0(top_line)
        bl = extrapolate_to_x0(bottom_line)

        # Giới hạn trong ảnh
        tl[1] = np.clip(tl[1], 0, h - 1)
        bl[1] = np.clip(bl[1], 0, h - 1)

        result = {
            "top-left": tl,
            "top-right": tr,
            "bot-right": br,
            "bot-left": bl
        }

        return result, "2/4 - right corners + Hough border"


    # =====================================================
    # Fallback: nếu Hough không tìm được đủ 2 cạnh
    # =====================================================

    print("⚠ Không tìm đủ 2 cạnh bằng Hough.")
    print("→ Dùng hình học + mép trái ảnh.")

    # Vector cạnh phải
    right_edge = br - tr

    right_height = np.linalg.norm(right_edge)

    # Tỷ lệ width/height linh hoạt hơn 1.586
    # do ảnh có perspective.
    estimated_width = right_height * 1.65

    # vector ngang gần vuông góc với cạnh phải
    v = np.array([
        right_edge[1],
        -right_edge[0]
    ], dtype=np.float32)

    v /= np.linalg.norm(v)

    # Hướng về bên trái
    if v[0] > 0:
        v = -v

    tl = tr + v * estimated_width
    bl = br + v * estimated_width

    # Nếu chưa tới mép trái thì ép về x=0
    tl[0] = 0
    bl[0] = 0

    return {
        "top-left": tl,
        "top-right": tr,
        "bot-right": br,
        "bot-left": bl
    }, "2/4 - right corners + geometry fallback"


def complete_card_points(image, detections):

    points = {}

    for name, info in detections.items():

        if name not in CORNER_NAMES:
            continue

        x1, y1, x2, y2 = map(
            float,
            info["box"]
        )

        points[name] = np.array(
            [
                (x1 + x2) / 2,
                (y1 + y2) / 2
            ],
            dtype=np.float32
        )

    n = len(points)

    print(f"YOLO detected: {n}/4")

    # -------------------------
    # 4/4
    # -------------------------

    if n == 4:
        #return points, "4/4 - YOLO"
        pts = np.array(
        list(points.values()),
        dtype=np.float32)

        # -----------------------------------------
        # Sort theo Y
        # -----------------------------------------

        idx_y = np.argsort(pts[:, 1])
        top = pts[idx_y[:2]]
        bottom = pts[idx_y[2:]]

        # -----------------------------------------
        # Top: trái -> phải
        # -----------------------------------------

        top = top[np.argsort(top[:, 0])]

        # -----------------------------------------
        # Bottom: trái -> phải
        # -----------------------------------------

        bottom = bottom[np.argsort(bottom[:, 0])]

        tl = top[0]
        tr = top[1]

        bl = bottom[0]
        br = bottom[1]

        corrected_points = {
          "top-left": tl,
          "top-right": tr,
          "bot-right": br,
          "bot-left": bl
        }

        return corrected_points, "4/4 - YOLO + geometric reorder"

    # -------------------------
    # 3/4
    # -------------------------

    if n == 3:

        points = complete_3_points(points)

        return points, "3/4 - inferred"

    # -------------------------
    # 2/4
    # -------------------------

    if n == 2:

        completed = complete_2_points_by_edges(
            image,
            points
        )

        if set(points.keys()) == {"top-right", "bot-right"}:

            return complete_right_corners(image, points)

        if completed is not None:

            return completed, "2/4 - Hough edges"

        raise ValueError(
            "Không tìm được 2 cạnh còn lại của card."
        )

    raise ValueError(
        f"Chỉ phát hiện {n}/4 điểm."
    )


# ==========================================================
# 3/4
# ==========================================================

def complete_3_points(points):

    points = {
        k: np.array(v, dtype=np.float32)
        for k, v in points.items()
    }

    missing = [
        x for x in CORNER_NAMES
        if x not in points
    ][0]

    tl = points.get("top-left")
    tr = points.get("top-right")
    br = points.get("bot-right")
    bl = points.get("bot-left")

    if missing == "top-left":

        points["top-left"] = (
            tr + bl - br
        )

    elif missing == "top-right":

        points["top-right"] = (
            tl + br - bl
        )

    elif missing == "bot-right":

        points["bot-right"] = (
            tr + bl - tl
        )

    elif missing == "bot-left":

        points["bot-left"] = (
            tl + br - tr
        )

    return points


# ==========================================================
# 2/4
# ==========================================================

def complete_2_points_by_edges(image, points):
    """
    Hoàn thiện 2 góc còn thiếu bằng cách:
    - tìm các cạnh của card bằng Canny + Hough
    - dùng 2 điểm YOLO làm điểm neo
    - kéo dài các cạnh tới giao điểm
    """

    h, w = image.shape[:2]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Làm ảnh dễ phát hiện cạnh hơn
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(
        gray,
        50,
        150
    )

    # Hough
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=80,
        minLineLength=min(w, h) // 8,
        maxLineGap=30
    )

    if lines is None:
        return None

    lines = lines.reshape(-1, 4)

    # ------------------------------------------------------
    # Trường hợp của bạn:
    #
    # bot-left + bot-right
    # ------------------------------------------------------

    if set(points.keys()) == {
        "bot-left",
        "bot-right"
    }:

        bl = points["bot-left"]
        br = points["bot-right"]

        bottom_vec = br - bl

        bottom_angle = np.arctan2(
            bottom_vec[1],
            bottom_vec[0]
        )

        bottom_length = np.linalg.norm(
            bottom_vec
        )

        if bottom_length == 0:
            return None

        # Hai cạnh bên phải gần vuông góc
        # với cạnh đáy
        target_angle = bottom_angle + np.pi / 2

        candidate_lines = []

        for line in lines:

            x1, y1, x2, y2 = map(
                float,
                line
            )

            p1 = np.array([x1, y1])
            p2 = np.array([x2, y2])

            vec = p2 - p1

            length = np.linalg.norm(vec)

            if length < 30:
                continue

            angle = np.arctan2(
                vec[1],
                vec[0]
            )

            # Chuẩn hóa góc về [-pi/2, pi/2]
            angle = (angle + np.pi/2) % np.pi - np.pi/2
            target = (target_angle + np.pi/2) % np.pi - np.pi/2

            angle_diff = abs(angle - target)

            if angle_diff > np.deg2rad(15):
                continue

            # Khoảng cách line tới BL
            line_vec = vec / length

            dist_bl = abs(
                np.cross(
                    line_vec,
                    bl - p1
                )
            )

            # Khoảng cách line tới BR
            dist_br = abs(
                np.cross(
                    line_vec,
                    br - p1
                )
            )

            dist = min(
                dist_bl,
                dist_br
            )

            candidate_lines.append(
                (dist, p1, p2)
            )

        if len(candidate_lines) < 2:
            return None

        # --------------------------------------------------
        # Tách line bên trái / bên phải
        # --------------------------------------------------

        left_lines = []
        right_lines = []

        for dist, p1, p2 in candidate_lines:

            center = (p1 + p2) / 2

            # So với trung điểm đáy
            if center[0] < (bl[0] + br[0]) / 2:
                left_lines.append(
                    (dist, p1, p2)
                )
            else:
                right_lines.append(
                    (dist, p1, p2)
                )

        if not left_lines or not right_lines:
            return None

        left_line = min(
            left_lines,
            key=lambda x: x[0]
        )

        right_line = min(
            right_lines,
            key=lambda x: x[0]
        )

        # --------------------------------------------------
        # Tạo đường thẳng vô hạn
        # --------------------------------------------------

        def line_from_points(p1, p2):

            x1, y1 = p1
            x2, y2 = p2

            A = y1 - y2
            B = x2 - x1
            C = x1 * y2 - x2 * y1

            return np.array(
                [A, B, C],
                dtype=np.float64
            )

        left = line_from_points(
            left_line[1],
            left_line[2]
        )

        right = line_from_points(
            right_line[1],
            right_line[2]
        )

        # --------------------------------------------------
        # Giao hai cạnh bên
        # --------------------------------------------------

        def intersection(l1, l2):

            cross = np.cross(l1, l2)

            if abs(cross[2]) < 1e-8:
                return None

            x = cross[0] / cross[2]
            y = cross[1] / cross[2]

            return np.array(
                [x, y],
                dtype=np.float32
            )

        # Hai cạnh bên giao nhau ở phía trên
        top_intersection = intersection(
            left,
            right
        )

        if top_intersection is None:
            return None

        # --------------------------------------------------
        # Nếu giao điểm nằm ngoài ảnh thì vẫn giữ nó
        # vì đó chính là trường hợp card bị cắt.
        # --------------------------------------------------

        # Tìm giao điểm của từng cạnh bên
        # với đường ngang qua top_intersection
        #
        # Nếu giao điểm quá xa / bất thường thì fallback.
        # --------------------------------------------------

        tl = top_intersection.copy()
        tr = top_intersection.copy()

        # --------------------------------------------------
        # Ước lượng 2 điểm trên cạnh trên
        # bằng cách dịch theo vector đáy
        # --------------------------------------------------

        top_width = bottom_length

        bottom_mid = (bl + br) / 2

        top_mid = top_intersection

        direction = bottom_vec / bottom_length

        tl = top_mid - direction * (top_width / 2)
        tr = top_mid + direction * (top_width / 2)

        points["top-left"] = tl
        points["top-right"] = tr

        return points

In [ ]:
#@title Fallback Detection

def order_quad(points):
    """
    Sắp xếp 4 điểm:
        TL -> TR -> BR -> BL
    """

    points = np.asarray(points, dtype=np.float32)

    result = np.zeros((4, 2), dtype=np.float32)

    s = points.sum(axis=1)
    d = np.diff(points, axis=1).reshape(-1)

    result[0] = points[np.argmin(s)]   # TL
    result[1] = points[np.argmin(d)]   # TR
    result[2] = points[np.argmax(s)]   # BR
    result[3] = points[np.argmax(d)]   # BL

    return result


def detect_card_fallback(image):
    """
    Fallback khi YOLO không phát hiện được corner.

    Sử dụng:
        HSV color segmentation
        +
        Canny edge
        +
        contour
        +
        hình học quadrilateral

    Trả về:
        points, score

    hoặc:
        None, None
    """

    h, w = image.shape[:2]
    image_area = h * w

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV
    )

    H, S, V = cv2.split(hsv)

    # =====================================================
    # 1. Tạo mask màu xanh nhạt của CCCD
    # =====================================================

    # CCCD thường có nhiều vùng cyan/xanh nhạt.
    #
    # Khoảng này cố tình khá rộng để chịu được
    # ánh sáng / white balance khác nhau.
    #
    mask_color = (
        (H >= 70) &
        (H <= 120) &
        (S >= 15) &
        (S <= 190) &
        (V >= 110)
    ).astype(np.uint8) * 255

    # =====================================================
    # 2. Morphology
    # =====================================================

    kernel_large = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (15, 15)
    )

    kernel_small = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    mask_color = cv2.morphologyEx(
        mask_color,
        cv2.MORPH_CLOSE,
        kernel_large,
        iterations=2
    )

    mask_color = cv2.morphologyEx(
        mask_color,
        cv2.MORPH_OPEN,
        kernel_small,
        iterations=1
    )

    # =====================================================
    # 3. Tìm contour màu
    # =====================================================

    contours_color, _ = cv2.findContours(
        mask_color,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours_color:

        area = cv2.contourArea(contour)

        # Không lấy vật thể quá nhỏ
        if area < image_area * 0.015:
            continue

        # Không lấy toàn bộ background
        if area > image_area * 0.80:
            continue

        perimeter = cv2.arcLength(
            contour,
            True
        )

        approx = cv2.approxPolyDP(
            contour,
            0.025 * perimeter,
            True
        )

        # Chấp nhận 4-6 điểm,
        # sau đó lấy minAreaRect nếu cần
        if len(approx) == 4:

            quad = approx.reshape(
                4, 2
            ).astype(np.float32)

        else:

            # Nếu contour không thành 4 điểm đẹp,
            # dùng minimum area rectangle
            rect = cv2.minAreaRect(contour)

            box = cv2.boxPoints(rect)

            quad = np.asarray(
                box,
                dtype=np.float32
            )

        # =================================================
        # Bounding box
        # =================================================

        x, y, bw, bh = cv2.boundingRect(
            quad.astype(np.int32)
        )

        if bw <= 0 or bh <= 0:
            continue

        ratio = max(bw, bh) / min(bw, bh)

        # CCCD có tỷ lệ khoảng 1.59.
        # Cho phép nghiêng/perspective mạnh.
        if ratio < 1.20 or ratio > 2.20:
            continue

        # =================================================
        # Rectangularity
        # =================================================

        quad_area = abs(
            cv2.contourArea(
                quad.astype(np.float32)
            )
        )

        if quad_area <= 0:
            continue

        rect_area = bw * bh

        rectangularity = quad_area / rect_area

        if rectangularity < 0.45:
            continue

        # =================================================
        # Tỷ lệ gần chuẩn CCCD
        # =================================================

        ratio_error = abs(
            ratio - 1.586
        ) / 1.586

        # =================================================
        # Score
        # =================================================

        area_score = min(
            area / (image_area * 0.30),
            1.0
        )

        score = (
            area_score * 0.45
            +
            rectangularity * 0.30
            +
            max(0, 1 - ratio_error) * 0.25
        )

        candidates.append(
            {
                "score": score,
                "quad": quad,
                "area": area,
                "ratio": ratio
            }
        )

    # =====================================================
    # 4. Không tìm được bằng màu
    # =====================================================

    if not candidates:

        # ---------------------------------------------
        # Fallback lần 2: Canny
        # ---------------------------------------------

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        gray = cv2.GaussianBlur(
            gray,
            (5, 5),
            0
        )

        edges = cv2.Canny(
            gray,
            50,
            150
        )

        edges = cv2.morphologyEx(
            edges,
            cv2.MORPH_CLOSE,
            kernel_large,
            iterations=2
        )

        contours, _ = cv2.findContours(
            edges,
            cv2.RETR_LIST,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:

            area = cv2.contourArea(contour)

            if area < image_area * 0.02:
                continue

            if area > image_area * 0.80:
                continue

            perimeter = cv2.arcLength(
                contour,
                True
            )

            approx = cv2.approxPolyDP(
                contour,
                0.02 * perimeter,
                True
            )

            if len(approx) != 4:
                continue

            quad = approx.reshape(
                4, 2
            ).astype(np.float32)

            x, y, bw, bh = cv2.boundingRect(
                quad.astype(np.int32)
            )

            if min(bw, bh) <= 0:
                continue

            ratio = max(bw, bh) / min(bw, bh)

            if ratio < 1.20 or ratio > 2.20:
                continue

            quad_area = abs(
                cv2.contourArea(quad)
            )

            rectangularity = (
                quad_area /
                (bw * bh)
            )

            if rectangularity < 0.50:
                continue

            ratio_error = abs(
                ratio - 1.586
            ) / 1.586

            score = (
                rectangularity * 0.6
                +
                max(0, 1 - ratio_error) * 0.4
            )

            candidates.append(
                {
                    "score": score,
                    "quad": quad,
                    "area": area,
                    "ratio": ratio
                }
            )

    # =====================================================
    # 5. Không tìm được
    # =====================================================

    if not candidates:
        return None, None

    # =====================================================
    # 6. Candidate tốt nhất
    # =====================================================

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    best = candidates[0]

    quad = order_quad(
        best["quad"]
    )

    points = {
        "top-left": quad[0],
        "top-right": quad[1],
        "bot-right": quad[2],
        "bot-left": quad[3]
    }

    return points, best["score"]

In [ ]:
#@title Validation

import numpy as np
import cv2


def validate_card_points(
    points,
    image_shape=None,
    min_area_ratio=0.01,
    min_point_distance=10,
    max_side_ratio=3.0,
    min_angle=15,
    max_angle=165,
    min_aspect=1.2,
    max_aspect=2.5,
    max_perspective_ratio=4.0,
    debug=True
):
    """
    Validate 4 corner points của CCCD.

    Không phụ thuộc orientation 0/90/180/270.

    Parameters
    ----------
    points : dict
        {
            'top-left':  [x, y],
            'top-right': [x, y],
            'bot-right': [x, y],
            'bot-left':  [x, y]
        }

    image_shape : tuple hoặc None
        shape của ảnh, ví dụ:
        image.shape

    min_area_ratio :
        Diện tích card tối thiểu / diện tích ảnh.

    min_point_distance :
        Khoảng cách tối thiểu giữa 2 corner.

    max_side_ratio :
        Tỷ lệ cạnh đối diện tối đa.

    min_angle / max_angle :
        Giới hạn góc của tứ giác.

    min_aspect / max_aspect :
        Tỷ lệ dài/rộng của CCCD.

    max_perspective_ratio :
        Mức perspective distortion tối đa.

    Returns
    -------
    valid : bool
    info : dict
    """

    required = [
        'top-left',
        'top-right',
        'bot-right',
        'bot-left'
    ]

    info = {
        "valid": False,
        "reason": None
    }

    # =========================================================
    # 1. CHECK ĐỦ POINT
    # =========================================================

    if not isinstance(points, dict):
        info["reason"] = "points không phải dict"
        return False, info

    missing = [k for k in required if k not in points]

    if missing:
        info["reason"] = f"Thiếu point: {missing}"
        return False, info

    # =========================================================
    # 2. CONVERT TO NUMPY
    # =========================================================

    try:
        pts = np.array(
            [points[k] for k in required],
            dtype=np.float32
        )
    except Exception as e:

        info["reason"] = f"Không convert được points: {e}"
        return False, info

    if pts.shape != (4, 2):

        info["reason"] = f"Shape không hợp lệ: {pts.shape}"
        return False, info

    # =========================================================
    # 3. NaN / INF
    # =========================================================

    if not np.isfinite(pts).all():

        info["reason"] = "Có NaN hoặc Inf"
        return False, info

    # =========================================================
    # 4. CHECK POINT TRÙNG
    # =========================================================

    distances = []

    for i in range(4):

        for j in range(i + 1, 4):

            d = np.linalg.norm(pts[i] - pts[j])

            distances.append(d)

            if d < min_point_distance:

                info["reason"] = (
                    f"2 point quá gần nhau: "
                    f"{required[i]} - {required[j]} "
                    f"({d:.2f}px)"
                )

                return False, info

    # =========================================================
    # 5. CONVEXITY
    # =========================================================

    contour = pts.reshape((-1, 1, 2))

    convex = cv2.isContourConvex(contour)

    if not convex:

        info["reason"] = "4 point không tạo thành tứ giác lồi"

        return False, info

    # =========================================================
    # 6. AREA
    # =========================================================

    area = cv2.contourArea(contour)

    if area <= 0:

        info["reason"] = f"Diện tích <= 0: {area}"

        return False, info

    # Nếu biết kích thước ảnh
    if image_shape is not None:

        image_h, image_w = image_shape[:2]

        image_area = image_w * image_h

        area_ratio = area / image_area

        if area_ratio < min_area_ratio:

            info["reason"] = (
                f"Card quá nhỏ: "
                f"{area_ratio:.4f} "
                f"< {min_area_ratio}"
            )

            return False, info

    else:

        area_ratio = None

    # =========================================================
    # 7. EDGE LENGTH
    # =========================================================

    edges = np.array([
        np.linalg.norm(pts[1] - pts[0]),  # TL -> TR
        np.linalg.norm(pts[2] - pts[1]),  # TR -> BR
        np.linalg.norm(pts[3] - pts[2]),  # BR -> BL
        np.linalg.norm(pts[0] - pts[3])   # BL -> TL
    ])

    top = edges[0]
    right = edges[1]
    bottom = edges[2]
    left = edges[3]

    # =========================================================
    # 8. CẠNH ĐỐI DIỆN
    # =========================================================

    width_ratio = max(top, bottom) / min(top, bottom)

    height_ratio = max(left, right) / min(left, right)

    if width_ratio > max_side_ratio:

        info["reason"] = (
            f"2 cạnh ngang quá chênh: "
            f"{width_ratio:.2f}"
        )

        return False, info

    if height_ratio > max_side_ratio:

        info["reason"] = (
            f"2 cạnh dọc quá chênh: "
            f"{height_ratio:.2f}"
        )

        return False, info

    # =========================================================
    # 9. ANGLES
    # =========================================================

    def angle_between(a, b, c):

        ba = a - b
        bc = c - b

        norm_ba = np.linalg.norm(ba)
        norm_bc = np.linalg.norm(bc)

        if norm_ba == 0 or norm_bc == 0:
            return 0

        cosine = np.dot(ba, bc) / (
            norm_ba * norm_bc
        )

        cosine = np.clip(cosine, -1.0, 1.0)

        return np.degrees(
            np.arccos(cosine)
        )

    angles = np.array([
        angle_between(
            pts[3], pts[0], pts[1]
        ),
        angle_between(
            pts[0], pts[1], pts[2]
        ),
        angle_between(
            pts[1], pts[2], pts[3]
        ),
        angle_between(
            pts[2], pts[3], pts[0]
        )
    ])

    if np.any(angles < min_angle):

        info["reason"] = (
            f"Có góc quá nhọn: "
            f"{angles.round(2)}"
        )

        return False, info

    if np.any(angles > max_angle):

        info["reason"] = (
            f"Có góc quá tù: "
            f"{angles.round(2)}"
        )

        return False, info

    # =========================================================
    # 10. ASPECT RATIO
    # =========================================================

    avg_width = (top + bottom) / 2
    avg_height = (left + right) / 2

    aspect = max(
        avg_width / avg_height,
        avg_height / avg_width
    )

    if not (
        min_aspect <= aspect <= max_aspect
    ):

        info["reason"] = (
            f"Aspect ratio bất thường: "
            f"{aspect:.2f}"
        )

        return False, info

    # =========================================================
    # 11. PERSPECTIVE DISTORTION
    # =========================================================

    # So sánh 2 đường chéo
    diagonal1 = np.linalg.norm(pts[2] - pts[0])
    diagonal2 = np.linalg.norm(pts[3] - pts[1])

    diagonal_ratio = (
        max(diagonal1, diagonal2)
        /
        min(diagonal1, diagonal2)
    )

    if diagonal_ratio > max_perspective_ratio:

        info["reason"] = (
            f"Perspective quá mạnh: "
            f"diagonal ratio = "
            f"{diagonal_ratio:.2f}"
        )

        return False, info

    # =========================================================
    # 12. SUCCESS
    # =========================================================

    info.update({
        "valid": True,
        "area": float(area),
        "area_ratio": (
            float(area_ratio)
            if area_ratio is not None
            else None
        ),

        "edges": {
            "top": float(top),
            "right": float(right),
            "bottom": float(bottom),
            "left": float(left)
        },

        "width_ratio": float(width_ratio),
        "height_ratio": float(height_ratio),

        "angles": angles.tolist(),

        "aspect_ratio": float(aspect),

        "diagonal_ratio": float(
            diagonal_ratio
        )
    })

    return True, info

# def validate_card_points(points, image_shape):

#     h, w = image_shape[:2]

#     tl = np.array(points["top-left"], dtype=np.float32)
#     tr = np.array(points["top-right"], dtype=np.float32)
#     br = np.array(points["bot-right"], dtype=np.float32)
#     bl = np.array(points["bot-left"], dtype=np.float32)

#     # -----------------------------
#     # 1. Không được NaN / Inf
#     # -----------------------------
#     all_points = np.array([tl, tr, br, bl])

#     if not np.all(np.isfinite(all_points)):
#         return False, "NaN/Inf"

#     # -----------------------------
#     # 2. Diện tích quadrilateral
#     # -----------------------------
#     area = cv2.contourArea(all_points)

#     if area < 1000:
#         return False, f"Area quá nhỏ: {area}"

#     # -----------------------------
#     # 3. Chiều rộng / chiều cao
#     # -----------------------------
#     width_top = np.linalg.norm(tr - tl)
#     width_bottom = np.linalg.norm(br - bl)

#     height_left = np.linalg.norm(bl - tl)
#     height_right = np.linalg.norm(br - tr)

#     if width_top < 50 or width_bottom < 50:
#         return False, "Width quá nhỏ"

#     if height_left < 50 or height_right < 50:
#         return False, "Height quá nhỏ"

#     # -----------------------------
#     # 4. Hai cạnh trên/dưới
#     # phải tương đối song song
#     # -----------------------------
#     v_top = tr - tl
#     v_bottom = br - bl

#     cross_horizontal = abs(
#         v_top[0] * v_bottom[1] - v_top[1] * v_bottom[0]
#     )

#     norm_horizontal = (
#         np.linalg.norm(v_top) *
#         np.linalg.norm(v_bottom)
#     )

#     if norm_horizontal > 0:
#         parallel_error = (
#             cross_horizontal / norm_horizontal
#         )

#         if parallel_error > 0.35:
#             return False, "Hai cạnh ngang không hợp lý"

#     # -----------------------------
#     # 5. Hai cạnh trái/phải
#     # -----------------------------
#     v_left = bl - tl
#     v_right = br - tr

#     cross_vertical = abs(
#         v_left[0] * v_right[1] - v_left[1] * v_right[0]
#     )

#     norm_vertical = (
#         np.linalg.norm(v_left) *
#         np.linalg.norm(v_right)
#     )

#     if norm_vertical > 0:

#         parallel_error = (
#             cross_vertical / norm_vertical
#         )

#         if parallel_error > 0.35:
#             return False, "Hai cạnh dọc không hợp lý"

#     return True, "OK"

def validate_crop_quality(crop, expected_ratio=1.586):
    if crop is None:
        return False, -999, "crop=None"

    h, w = crop.shape[:2]

    if h < 100 or w < 100:
        return False, -999, "crop quá nhỏ"

    ratio = w / h

    # CCCD phải ưu tiên landscape
    ratio_score = np.exp(
        -((ratio - expected_ratio) / 0.35) ** 2
    )

    # Phạt mạnh nếu thành portrait
    if ratio < 1.0:
        return False, ratio_score, f"portrait ratio={ratio:.2f}"

    # Phạt crop quá dài / quá dẹt
    if ratio > 2.2:
        return False, ratio_score, f"ratio quá lớn={ratio:.2f}"

    return True, ratio_score, f"ratio={ratio:.2f}"

In [ ]:
#@title Find Best Candidate

def rotation_score(num_points, quality, confidence):
    return (
        num_points * 100
        + quality * 20
        + confidence * 5
    )

def rotate(image, angle):
    if angle == 0:
        return image.copy()

    elif angle == 90:
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)

    elif angle == 180:
        return cv2.rotate(image, cv2.ROTATE_180)

    elif angle == 270:
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)

    else:
        raise ValueError(
            "angle phải là 0, 90, 180 hoặc 270"
        )

def crop_orientation_score(crop):

    h, w = crop.shape[:2]

    if h == 0 or w == 0:
        return -999

    ratio = w / h

    # CCCD ID-1 ≈ 1.586
    target = 1.586

    error = abs(
        ratio - target
    )

    score = 100 - error * 50

    return score


def try_card_candidate(
    output_image,
    work_image,
    angle,
    preprocess_name,
    model_v5,
    expected_ratio=1.586,
    debug=False
):
    detections = get_detections(model_v5, work_image)

    n = len(detections)

    if n < 2:
        return None

    conf_sum = sum(
        d["conf"] for d in detections.values()
    )

    # ==========================================
    # Complete 4 points
    # ==========================================

    try:
        points, method = complete_card_points(
            work_image,
            detections
        )
    except Exception as e:
        if debug:
            print(f"    complete lỗi: {e}")
        return None

    if points is None:
        return None

    # ==========================================
    # Validate points
    # ==========================================

    valid, reason = validate_card_points(
        points,
        work_image.shape,
        debug=debug
    )

    if not valid:
        if debug:
            print(f"    Point invalid: {reason}")
        return None

    # ==========================================
    # Crop từ ảnh màu gốc
    # ==========================================

    try:
        crop = perspective_crop(
            output_image,
            points
        )
    except Exception as e:
        if debug:
            print(f"    Crop lỗi: {e}")
        return None

    if crop is None:
        return None

    # ==========================================
    # Crop quality
    # ==========================================

    quality = evaluate_crop_quality(
        crop,
        points,
        output_image.shape,
        expected_ratio=expected_ratio
    )

    if debug:
        print(
            f"    {preprocess_name:8s} "
            f"{angle:3d}° | "
            f"{n}/4 | "
            f"conf={conf_sum:.3f} | "
            f"quality={quality['score']:.3f} | "
            f"{quality['reason']}"
        )

    if not quality["valid"]:
        return None

    # ==========================================
    # Score
    # ==========================================

    detection_score = n / 4

    confidence_score = min(
        conf_sum / 4.0,
        1.0
    )

    total_score = (
        quality["score"] * 0.75 +
        detection_score * 0.15 +
        confidence_score * 0.10
    )

    # Ưu tiên nhẹ ảnh gốc
    if preprocess_name == "original":
        total_score += 0.03

    # Ưu tiên nhẹ 0°
    if angle == 0:
        total_score += 0.05

    return {
        "angle": angle,
        "preprocess": preprocess_name,

        # ảnh màu gốc sau rotate
        "image": output_image,

        # ảnh dùng detect
        "work_image": work_image,

        "detections": detections,
        "points": points,
        "method": method,

        "crop": crop,

        "num_detections": n,
        "conf_sum": conf_sum,

        "quality": quality,
        "score": total_score
    }

def find_best_card(
    original,
    model_v5,
    expected_ratio=1.586,
    debug=True
):
    angles = [0, 90, 180, 270]

    # ============================================================
    # HÀM XẾP HẠNG CANDIDATE
    # ============================================================

    def candidate_rank(candidate):
        """
        Thứ tự ưu tiên:

        1. Số điểm góc detect được: 4/4 > 3/4 > 2/4
        2. Quality / score
        3. Confidence YOLO
        """

        num_points = candidate.get(
            "num_points",
            candidate.get("detected_count", 0)
        )

        score = candidate.get(
            "score",
            0
        )

        confidence = candidate.get(
            "conf",
            candidate.get("confidence", 0)
        )

        return (
            num_points,
            score,
            confidence
        )

    # ============================================================
    # STAGE 1 — ORIGINAL
    # ============================================================

    if debug:
        print("\n" + "=" * 50)
        print("STAGE 1 — ORIGINAL")
        print("=" * 50)

    candidates = []

    for angle in angles:

        rotated = rotate(
            original,
            angle
        )

        candidate = try_card_candidate(
            output_image=rotated,
            work_image=rotated,
            angle=angle,
            preprocess_name="original",
            model_v5=model_v5,
            expected_ratio=expected_ratio,
            debug=debug
        )

        if candidate is not None:

            # ----------------------------------------------------
            # Lưu số điểm detect nếu candidate chưa có
            # ----------------------------------------------------

            if "num_points" not in candidate:

                if "detections" in candidate:
                    candidate["num_points"] = len(
                        candidate["detections"]
                    )

                elif "points" in candidate:
                    candidate["num_points"] = len(
                        candidate["points"]
                    )

                elif "detected_count" in candidate:
                    candidate["num_points"] = candidate[
                        "detected_count"
                    ]

            candidates.append(candidate)

    # ============================================================
    # CHỌN ORIGINAL TỐT NHẤT
    # ============================================================

    if candidates:

        best = max(
            candidates,
            key=candidate_rank
        )

        if debug:

            print("\n✓ Original rotation hợp lệ")

            print(
                f"→ Selected: "
                f"{best['angle']}°"
            )

            print(
                f"→ Points: "
                f"{best.get('num_points', '?')}/4"
            )

            print(
                f"→ Score: "
                f"{best.get('score', 0):.3f}"
            )

            print(
                f"→ Confidence: "
                f"{best.get('conf', best.get('confidence', 0)):.3f}"
            )

        return best

    # ============================================================
    # STAGE 2 — PREPROCESSING RESCUE
    # ============================================================

    if debug:
        print("\n" + "=" * 50)
        print("STAGE 2 — PREPROCESSING RESCUE")
        print("=" * 50)

    rescue_candidates = []

    for preprocess_name in [
        "clahe",
        "sharpen"
    ]:

        for angle in angles:

            # ----------------------------------------------------
            # Ảnh màu gốc
            # ----------------------------------------------------

            output_image = rotate(
                original,
                angle
            )

            # ----------------------------------------------------
            # Tạo ảnh detection
            # ----------------------------------------------------

            if preprocess_name == "clahe":

                gray = cv2.cvtColor(
                    output_image,
                    cv2.COLOR_BGR2GRAY
                )

                clahe = cv2.createCLAHE(
                    clipLimit=2.0,
                    tileGridSize=(8, 8)
                )

                enhanced = clahe.apply(
                    gray
                )

                work_image = cv2.cvtColor(
                    enhanced,
                    cv2.COLOR_GRAY2BGR
                )

            elif preprocess_name == "sharpen":

                kernel = np.array([
                    [0, -1, 0],
                    [-1, 5, -1],
                    [0, -1, 0]
                ], dtype=np.float32)

                work_image = cv2.filter2D(
                    output_image,
                    -1,
                    kernel
                )

            # ----------------------------------------------------
            # Detect
            # ----------------------------------------------------

            candidate = try_card_candidate(
                output_image=output_image,
                work_image=work_image,
                angle=angle,
                preprocess_name=preprocess_name,
                model_v5=model_v5,
                expected_ratio=expected_ratio,
                debug=debug
            )

            if candidate is not None:

                # ------------------------------------------------
                # Lưu số điểm detect
                # ------------------------------------------------

                if "num_points" not in candidate:

                    if "detections" in candidate:
                        candidate["num_points"] = len(
                            candidate["detections"]
                        )

                    elif "points" in candidate:
                        candidate["num_points"] = len(
                            candidate["points"]
                        )

                    elif "detected_count" in candidate:
                        candidate["num_points"] = candidate[
                            "detected_count"
                        ]

                rescue_candidates.append(
                    candidate
                )

    # ============================================================
    # CHỌN RESCUE TỐT NHẤT
    # ============================================================

    if rescue_candidates:

        best = max(
            rescue_candidates,
            key=candidate_rank
        )

        if debug:

            print(
                "\n✓ Preprocessing rescue thành công"
            )

            print(
                f"→ {best.get('preprocess_name', best.get('preprocess', '?'))} "
                f"| {best['angle']}° "
                f"| points={best.get('num_points', '?')}/4 "
                f"| score={best.get('score', 0):.3f}"
            )

        return best

    # ============================================================
    # FAIL
    # ============================================================

    if debug:
        print(
            "\n✗ Không tìm được candidate hợp lệ."
        )

    return None

In [ ]:
#@title Preprocessing

def evaluate_crop_quality(
    crop,
    points,
    image_shape,
    expected_ratio=1.586
):
    if crop is None:
        return {
            "valid": False,
            "score": -999,
            "reason": "crop=None"
        }

    h, w = crop.shape[:2]

    if h < 100 or w < 100:
        return {
            "valid": False,
            "score": -999,
            "reason": "crop quá nhỏ"
        }

    # ==========================================
    # 1. Tỷ lệ crop
    # ==========================================

    ratio = w / h

    # CCCD ưu tiên landscape
    if ratio < 1.0:
        return {
            "valid": False,
            "score": -999,
            "reason": f"portrait ratio={ratio:.3f}"
        }

    if ratio > 2.2:
        return {
            "valid": False,
            "score": -999,
            "reason": f"ratio quá lớn={ratio:.3f}"
        }

    ratio_error = abs(ratio - expected_ratio)

    ratio_score = np.exp(
        -((ratio_error / 0.25) ** 2)
    )

    # ==========================================
    # 2. Kiểm tra 4 cạnh
    # ==========================================

    top, right, bottom, left = edge_lengths(points)

    if min(top, right, bottom, left) < 20:
        return {
            "valid": False,
            "score": -999,
            "reason": "có cạnh quá ngắn"
        }

    # top ≈ bottom
    horizontal_error = abs(top - bottom) / max(top, bottom)

    # left ≈ right
    vertical_error = abs(left - right) / max(left, right)

    edge_error = (
        horizontal_error +
        vertical_error
    ) / 2

    edge_score = np.exp(
        -((edge_error / 0.30) ** 2)
    )

    # ==========================================
    # 3. Tỷ lệ hình học của card
    # ==========================================

    avg_width = (top + bottom) / 2
    avg_height = (left + right) / 2

    if avg_height <= 0:
        return {
            "valid": False,
            "score": -999,
            "reason": "height không hợp lệ"
        }

    geometric_ratio = avg_width / avg_height

    geo_error = abs(
        geometric_ratio - expected_ratio
    )

    geo_ratio_score = np.exp(
        -((geo_error / 0.35) ** 2)
    )

    # ==========================================
    # 4. Diện tích quadrilateral
    # ==========================================

    area = quad_area(points)

    img_h, img_w = image_shape[:2]
    image_area = img_h * img_w

    area_ratio = area / image_area

    # Card quá nhỏ hoặc chiếm gần toàn bộ ảnh
    if area_ratio < 0.01:
        return {
            "valid": False,
            "score": -999,
            "reason": f"card quá nhỏ area={area_ratio:.4f}"
        }

    if area_ratio > 0.98:
        return {
            "valid": False,
            "score": -999,
            "reason": f"card chiếm gần toàn ảnh area={area_ratio:.4f}"
        }

    # ==========================================
    # Tổng điểm
    # ==========================================

    score = (
        ratio_score * 0.40 +
        edge_score * 0.30 +
        geo_ratio_score * 0.25 +
        min(area_ratio / 0.20, 1.0) * 0.05
    )

    # Ngưỡng an toàn
    valid = (
        ratio_score >= 0.25 and
        edge_score >= 0.20 and
        geo_ratio_score >= 0.25
    )

    return {
        "valid": valid,
        "score": float(score),
        "reason": (
            f"ratio={ratio:.3f}, "
            f"geo_ratio={geometric_ratio:.3f}, "
            f"edge_error={edge_error:.3f}, "
            f"area={area_ratio:.3f}"
        ),
        "ratio": ratio,
        "geometric_ratio": geometric_ratio,
        "edge_error": edge_error,
        "area_ratio": area_ratio
    }

def make_preprocess_variants(image):
    variants = {}

    # ==========================================
    # Original
    # ==========================================

    variants["original"] = image.copy()

    # ==========================================
    # CLAHE
    # ==========================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(gray)

    variants["clahe"] = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2BGR
    )

    # ==========================================
    # Sharpen
    # ==========================================

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ], dtype=np.float32)

    variants["sharpen"] = cv2.filter2D(
        image,
        -1,
        kernel
    )

    return variants

In [ ]:
#@title OCR

import re
import unicodedata

keywords = {
        "socialist republic": 4,
        "republic of vietnam": 4,
        "identity card": 4,
        "full name": 3,
        "date of birth": 3,
        "sex": 2,
        "nationality": 2,
        "place of origin": 3,
        "place of residence": 3,
        "personal identification": 4,
        "idvnm": 4
    }


def normalize_ocr_text(text):
    text = text.lower()

    text = unicodedata.normalize("NFD", text)
    text = "".join(
        c for c in text
        if unicodedata.category(c) != "Mn"
    )

    # Chuẩn hóa whitespace
    text = " ".join(text.split())

    return text


def score_cccd_ocr(text, debug=False):
    t = normalize_ocr_text(text)

    score = 0
    matched = []

    for keyword, weight in keywords.items():
        keyword_norm = normalize_ocr_text(keyword)

        if keyword_norm in t:
            score += weight
            matched.append((keyword, weight))

    digits = sum(c.isdigit() for c in t)

    if digits >= 6:
        score += 1
        matched.append(("6+ digits", 1))

    if digits >= 12:
        score += 2
        matched.append(("12+ digits", 2))


    return score


def my_ocr_function(crop):
    """
    Thay phần bên trong bằng OCR Google Docs,
    VietOCR hoặc OCR engine bạn đang dùng.
    """

    text = ""
    confidence = 0.0

    # TODO: gọi OCR thật tại đây

    # Điểm tạm thời, cần thay bằng kết quả OCR thực tế
    score = confidence

    return {
        "score": score,
        "text": text,
        "confidence": confidence
    }


def get_ocr_words(image):
    data = pytesseract.image_to_data(
        image,
        lang=lang,
        config=config,
        output_type=pytesseract.Output.DICT
    )

    words = []

    for i, text in enumerate(data["text"]):
        text = text.strip()

        if not text:
            continue

        try:
            conf = float(data["conf"][i])
        except:
            conf = -1

        words.append({
            "text": text,
            "conf": conf,
            "x": data["left"][i],
            "y": data["top"][i],
            "w": data["width"][i],
            "h": data["height"][i],
        })

    return words

def decide_ocr_orientation(
    score_0,
    score_180,
    min_diff=3,
    min_ratio=1.5
):

    diff = abs(score_0 - score_180)

    best = max(
        score_0,
        score_180
    )

    worst = min(
        score_0,
        score_180
    )

    ratio = (
        best / max(worst, 1)
    )

    # Không đủ tín hiệu
    if diff < min_diff:
        return None

    # Điểm cao hơn phải đủ vượt trội
    if ratio < min_ratio:
        return None

    if score_0 > score_180:
        return 0

    return 180

def auto_orient_180(crop, lang, config, debug=True):

    # =========================
    # 0°
    # =========================

    ocr_words_0 = get_ocr_words(crop)
    full_text_0 = " ".join([word['text'] for word in ocr_words_0])
    score_0 = score_cccd_ocr(full_text_0, debug)

    # =========================
    # 180°
    # =========================

    crop_180 = cv2.rotate(
        crop,
        cv2.ROTATE_180
    )

    ocr_words_180 = get_ocr_words(crop_180)
    full_text_180 = " ".join([word['text'] for word in ocr_words_180])
    score_180 = score_cccd_ocr(full_text_180, debug)

    if debug:
        print(f"OCR 0°   : {score_0}")
        print(f"OCR 180° : {score_180}")

    # =========================
    # Chọn hướng
    # =========================

    # angle = decide_ocr_orientation(
    #     score_0,
    #     score_180,
    #     min_diff=3,
    #     min_ratio=1.5
    # )

    # if angle is None:
    #     return None

    # if angle == 0:
    #     return {
    #         "crop": crop,
    #         "angle": 0,
    #         "text": full_text_0,
    #         "score": score_0
    #     }

    # return {
    #     "crop": crop_180,
    #     "angle": 180,
    #     "text": full_text_180,
    #     "score": score_180
    # }

    if score_180 > score_0:

        if debug:
            print("→ Chọn 180°")

            return {
                "crop": crop_180,
                "angle": 180,
                #"text": text_180,
                "score": score_180
            }

    print("→ Chọn 0°")
    return {
        "crop": crop,
        "angle": 0,
        #"text": text_0,
        "score": score_0
    }

In [ ]:
#@title Face, QR, MRZ

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


def detect_face_score(crop, debug=False):

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40)
    )

    if len(faces) == 0:
        return 0, None

    h, w = crop.shape[:2]

    best_score = 0
    best_face = None

    for (x, y, fw, fh) in faces:

        cx = x + fw / 2
        cy = y + fh / 2

        # Tỉ lệ kích thước khuôn mặt so với thẻ
        area_ratio = (fw * fh) / (w * h)

        score = 0

        # Khuôn mặt phải có kích thước hợp lý
        if 0.005 < area_ratio < 0.25:
            score += 2

        # Không nằm sát mép
        if (
            x > 0.02 * w and
            y > 0.02 * h and
            x + fw < 0.98 * w and
            y + fh < 0.98 * h
        ):
            score += 1

        if score > best_score:
            best_score = score
            best_face = (x, y, fw, fh)

    if debug:
        print("Faces:", len(faces))
        print("Best face:", best_face)
        print("Face score:", best_score)

    return best_score, best_face

qr_detector = cv2.QRCodeDetector()


def detect_qr_score(crop, debug=False):

    try:
        ok, points = qr_detector.detect(crop)
    except Exception:
        return 0, None

    if not ok or points is None:
        return 0, None

    points = points.reshape(-1, 2)

    h, w = crop.shape[:2]

    x_min = points[:, 0].min()
    x_max = points[:, 0].max()
    y_min = points[:, 1].min()
    y_max = points[:, 1].max()

    qr_w = x_max - x_min
    qr_h = y_max - y_min

    if qr_w <= 0 or qr_h <= 0:
        return 0, None

    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2

    score = 3

    # QR phải tương đối vuông
    ratio = qr_w / qr_h

    if 0.7 < ratio < 1.3:
        score += 2

    # Không quá nhỏ
    if qr_w > 0.05 * w:
        score += 1

    if debug:
        print("QR:", points)
        print("QR score:", score)

    return score, points

# def detect_mrz_score(crop, debug=False):

#     gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

#     # Tăng tương phản nhẹ
#     gray = cv2.GaussianBlur(gray, (3, 3), 0)

#     binary = cv2.adaptiveThreshold(
#         gray,
#         255,
#         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#         cv2.THRESH_BINARY_INV,
#         31,
#         9
#     )

#     h, w = gray.shape

#     # MRZ thường là các dòng nằm ngang
#     kernel = cv2.getStructuringElement(
#         cv2.MORPH_RECT,
#         (max(15, w // 30), 3)
#     )

#     horizontal = cv2.morphologyEx(
#         binary,
#         cv2.MORPH_CLOSE,
#         kernel
#     )

#     contours, _ = cv2.findContours(
#         horizontal,
#         cv2.RETR_EXTERNAL,
#         cv2.CHAIN_APPROX_SIMPLE
#     )

#     candidates = []

#     for cnt in contours:

#         x, y, cw, ch = cv2.boundingRect(cnt)

#         ratio = cw / max(ch, 1)

#         # vùng dài ngang
#         if (
#             cw > 0.25 * w and
#             ratio > 5
#         ):
#             candidates.append(
#                 (x, y, cw, ch)
#             )

#     # Chỉ xem vùng phía dưới là MRZ candidate
#     bottom_candidates = [
#         r for r in candidates
#         if r[1] > 0.45 * h
#     ]

#     score = 0

#     if len(bottom_candidates) >= 1:
#         score += 1

#     if len(bottom_candidates) >= 2:
#         score += 2

#     if debug:
#         print("MRZ candidates:", bottom_candidates)
#         print("MRZ score:", score)

#     return score, bottom_candidates

def detect_mrz_score(crop, debug=False):

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    binary = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        9
    )

    h, w = gray.shape

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (max(15, w // 30), 3)
    )

    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        horizontal,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for cnt in contours:

        x, y, cw, ch = cv2.boundingRect(cnt)

        ratio = cw / max(ch, 1)

        if (
            cw > 0.35 * w
            and ratio > 8
            and 0.01 * h < ch < 0.12 * h
        ):
            candidates.append(
                (x, y, cw, ch)
            )

    # Chỉ lấy vùng phía dưới
    bottom_candidates = [
        r for r in candidates
        if r[1] > 0.55 * h
    ]

    score = 0

    # =====================================================
    # Phải có ít nhất 2 dòng
    # =====================================================

    if len(bottom_candidates) >= 2:

        # sắp xếp theo Y
        bottom_candidates = sorted(
            bottom_candidates,
            key=lambda r: r[1]
        )

        # tìm cặp dòng gần nhau
        for i in range(
            len(bottom_candidates) - 1
        ):

            r1 = bottom_candidates[i]
            r2 = bottom_candidates[i + 1]

            x1, y1, w1, h1 = r1
            x2, y2, w2, h2 = r2

            center1 = y1 + h1 / 2
            center2 = y2 + h2 / 2

            gap = abs(center2 - center1)

            # Hai dòng phải tương đối gần nhau
            if gap < 0.15 * h:

                # Độ rộng tương đối giống nhau
                width_ratio = min(w1, w2) / max(w1, w2)

                if width_ratio > 0.65:

                    score = 5
                    break

    if debug:
        print(
            "MRZ candidates:",
            bottom_candidates
        )

        print(
            "MRZ score:",
            score
        )

    return score, bottom_candidates



def FQM_orientation_score(crop, debug=False):
    """
    Chấm điểm nhanh bằng Face + QR + MRZ.
    """

    face_score, _ = detect_face_score(crop, debug=False)
    qr_score, _ = detect_qr_score(crop, debug=False)
    mrz_score, _ = detect_mrz_score(crop, debug=False)

    # Có thể điều chỉnh sau khi kiểm tra dữ liệu thực tế
    total_score = (
        face_score * 4.0 +
        qr_score * 2.0)

    if debug:
        print(
            f"Face={face_score}, "
            f"QR={qr_score}, "
            f"MRZ={mrz_score}, "
            f"Total={total_score:.2f}"
        )

    return {
        "score": total_score,
        "face": face_score,
        "qr": qr_score,
        "mrz": mrz_score
    }


def auto_orient_cascade(
    crop,
    ocr_function=None,
    fast_threshold=5.0,
    ocr_threshold=5.0,
    debug=True
):
    """
    Stage 2:

    - F/Q/M được dùng trước.
    - Nếu F/Q/M không chắc chắn thì gọi OCR.
    - Không cần reference image.
    """

    if crop is None:
        print("❌ crop=None")
        return None

    crop_0 = crop.copy()

    crop_180 = cv2.rotate(
        crop,
        cv2.ROTATE_180
    )

    # =====================================================
    # LEVEL 1 — FAST DETECTORS
    # =====================================================

    fast_0 = FQM_orientation_score(
        crop_0,
        debug=False
    )

    fast_180 = FQM_orientation_score(
        crop_180,
        debug=False
    )

    score_0 = fast_0["score"]
    score_180 = fast_180["score"]

    difference = abs(score_0 - score_180)

    if debug:
        print("\n=== FAST ORIENTATION ===")
        print(
            f"0°   | Face={fast_0['face']} "
            f"QR={fast_0['qr']} "
            f"MRZ={fast_0['mrz']} "
            f"TOTAL={score_0:.2f}"
        )
        print(
            f"180° | Face={fast_180['face']} "
            f"QR={fast_180['qr']} "
            f"MRZ={fast_180['mrz']} "
            f"TOTAL={score_180:.2f}"
        )
        print(f"Difference FQM: {difference:.2f}")

    # =====================================================
    # FAST DECISION
    # =====================================================

    face_0 = fast_0["face"]
    face_180 = fast_180["face"]

    qr_0 = fast_0["qr"]
    qr_180 = fast_180["qr"]

    face_diff = abs(face_0 - face_180)
    qr_diff = abs(qr_0 - qr_180)

    # =====================================================
    # 1. FACE
    # =====================================================

    if (
        face_diff > 2
        and max(face_0, face_180) > 2
    ):

        angle = (
            0
            if face_0 > face_180
            else 180
        )

        final_crop = (
            crop_0
            if angle == 0
            else crop_180
        )

        if final_crop is not None:
            print(
                f"→ Face xác định orientation: {angle}°"
            )

            return {
                "crop": final_crop,
                "angle": angle,
                "method": "fast_face",
                "score": max(face_0, face_180),
                "score_0": score_0,
                "score_180": score_180,
                "fast_0": fast_0,
                "fast_180": fast_180,
                "ocr_used": False
            }


    # =====================================================
    # 2. QR
    # =====================================================

    # if (
    #     qr_diff >= 2
    #     and max(qr_0, qr_180) >= 3
    # ):

    #     angle = (
    #         0
    #         if qr_0 > qr_180
    #         else 180
    #     )

    #     final_crop = (
    #         crop_0
    #         if angle == 0
    #         else crop_180
    #     )

    #     if debug:
    #         print(
    #             f"→ QR xác định orientation: {angle}°"
    #         )

    #     return {
    #         "crop": final_crop,
    #         "angle": angle,
    #         "method": "fast_qr",
    #         "score": max(qr_0, qr_180),
    #         "score_0": score_0,
    #         "score_180": score_180,
    #         "fast_0": fast_0,
    #         "fast_180": fast_180,
    #         "ocr_used": False
    #     }


    # =====================================================
    # 3. FACE / QR không đủ rõ
    # =====================================================

    if debug:
        print(
            "→ Face/QR không đủ chắc chắn "
            "→ chuyển sang OCR"
        )


    # =====================================================
    # LEVEL 2 — OCR FALLBACK
    # =====================================================


    if ocr_function is None:
        if debug:
            print("⚠ Không có ocr_function → UNKNOWN")

        return {
            "crop": crop_0,
            "angle": None,
            "method": "unknown_no_ocr",
            "score": max(score_0, score_180),
            "score_0": score_0,
            "score_180": score_180,
            "fast_0": fast_0,
            "fast_180": fast_180,
            "ocr_used": False
        }

    ocr_res = auto_orient_180(crop_0, lang, config)
    ocr_score = ocr_res["score"]

    if debug:
        print("\n=== OCR FALLBACK ===")
        print(f"Best OCR: {ocr_score:.4f}")
        print(f"Rorate: {ocr_res["angle"]}")


    if ocr_score < ocr_threshold:
        return {
            "crop": None,
            "angle": None,
            "method": "unknown_ocr_tie",
            "score": ocr_score,
            "score_0": score_0,
            "score_180": score_180,

            "fast_0": fast_0,
            "fast_180": fast_180,
            "ocr_used": True
        }

    angle = ocr_res["angle"]

    # angle = (
    #     0
    #     if ocr_score <= 0
    #     else 180
    # )


    final_crop = (
        crop_0
        if angle == 0
        else crop_180
    )

    return {
        "crop": final_crop,
        "angle": angle,
        "method": "ocr_fallback",
        "score": ocr_score,
        "score_0": score_0,
        "score_180": score_180,
        # "ocr_score_0": ocr_score_0,
        # "ocr_score_180": ocr_score_180,
        "fast_0": fast_0,
        "fast_180": fast_180,
        "ocr_used": True
    }

In [ ]:
#@title Processing

def process_card(
    image,
    model_v5,
    crop=True,
    scale=1.0,
    debug=True,
    expand=False
):
    """
    Pipeline:

    1. YOLO detect corners
    2. Auto rotate CHỈ khi có >= 3 điểm
    3. Complete missing corners
    4. Validate
    5. Optional expand
    6. Perspective crop
    """

    original = image.copy()

    ocr_result = None
    ocr_score = None
    orientation = 0
    selected_angle = 0
    selected_preprocess = None

    # =====================================================
    # 1. YOLO trên ảnh gốc
    # =====================================================

    best = find_best_card(
        original,
        model_v5,
        expected_ratio=1.586,
        debug=debug)

    if best is None:
        print("\n❌ Không tìm được CCCD")

        return {
            "image": original,
            "crop": None,
            "points": None,
            "points_before_expand": None,
            "detections": None,
            "angle": None,
            "preprocess": None,
            "method": "failed",
            "debug_points_image": None,
            "debug_crop": None
        }

    selected_angle = best["angle"]
    selected_preprocess = best["preprocess"]

    #image_normalized = rotate(original, selected_angle)

    #text = pytesseract.image_to_string(original, lang = lang, config = config)
    image_normalized = best["image"]

    detections = best["detections"]
    n_original = len(detections)

    print(f"\nSelected: "
    f"{selected_preprocess} | "
    f"{selected_angle}° | "
    f"{len(detections)}/4 | "
    f"{image_normalized.shape}")

    # =====================================================
    # 2. XỬ LÝ THEO SỐ ĐIỂM
    # =====================================================

    # -----------------------------------------------------
    # CASE >= 2

    if n_original >= 2:

        points, method = complete_card_points(
            image_normalized,
            detections
        )

        print("Method:", method)

    # -----------------------------------------------------
    # CASE < 2
    # -----------------------------------------------------

    else:

        print("→ YOLO < 2 → fallback")

        image_normalized = original.copy()

        selected_angle = 0

        fallback_points, score = detect_card_fallback(
            image_normalized
        )

        if fallback_points is None:

            return {
                "image": image_normalized,
                "crop": None,
                "points": None,
                "points_before_expand": None,
                "detections": detections,
                "preprocess": selected_preprocess,
                "angle": selected_angle,
                "method": "failed",
                "debug_points_image": image_normalized.copy(),
                "debug_crop": None
                #"candidates": all_candidates
            }

        points = {
            "top-left": fallback_points[0],
            "top-right": fallback_points[1],
            "bot-right": fallback_points[2],
            "bot-left": fallback_points[3]
            # "top-left": np.array(
            #     fallback_points[0],
            #     dtype=np.float32
            # ),

            # "top-right": np.array(
            #     fallback_points[1],
            #     dtype=np.float32
            # ),

            # "bot-right": np.array(
            #     fallback_points[2],
            #     dtype=np.float32
            # ),

            # "bot-left": np.array(
            #     fallback_points[3],
            #     dtype=np.float32
            # )
        }

        method = "card fallback"

    # =====================================================
    # 3. Lưu points trước expand
    # =====================================================

    points_before_expand = {
        k: np.array(v, dtype=np.float32).copy()
        for k, v in points.items()
    }


    # =====================================================
    # 4. DEBUG
    # =====================================================

    if debug:

        debug_img = debug_points(
            image_normalized,
            points_before_expand
        )

        print("\n=== POINTS BEFORE EXPAND ===")

        for name in [
            "top-left",
            "top-right",
            "bot-right",
            "bot-left"
        ]:
            print(
                f"{name:12s}: "
                f"{points_before_expand[name]}"
            )

    else:

        debug_img = None


    # =====================================================
    # 5. Validate
    # =====================================================

    valid, info = validate_card_points(
        points_before_expand,
        image_normalized.shape,
        debug=debug
    )

    print(
        f"\nPoint validation: "
        f"{valid} | {info}"
    )

    if not valid:

        print(
            "⚠ Points không hợp lệ → "
            "không perspective crop."
        )

        return {
            "image": image_normalized,
            "crop": None,
            "points": points_before_expand,
            "points_before_expand": points_before_expand,
            "detections": detections,
            "angle": selected_angle,
            "method": method + " | INVALID POINTS",
            "debug_points_image": debug_img,
            "debug_crop": None
        }


    # =====================================================
    # 6. Expand nếu cần
    # =====================================================

    if expand:

        points = expand_card_points(
            points_before_expand,
            top_margin=0.02,
            bottom_margin=0.08,
            left_margin=0.01,
            right_margin=0.01
        )

        print("\n=== POINTS AFTER EXPAND ===")

        for name in [
            "top-left",
            "top-right",
            "bot-right",
            "bot-left"
        ]:
            print(
                f"{name:12s}: "
                f"{points[name]}"
            )

    else:

        # QUAN TRỌNG:
        # dùng nguyên points, không reorder
        points = points_before_expand


    # =====================================================
    # 7. Debug sau expand
    # =====================================================

    if debug and expand:

        debug_img = debug_points(
            image_normalized,
            points
        )


    # =====================================================
    # 8. Perspective crop
    # =====================================================

    card_crop = None

    if crop:

        print("\n=== PERSPECTIVE CROP ===")

        print("Image shape:",
              image_normalized.shape)

        print("Points:")

        for name in [
            "top-left",
            "top-right",
            "bot-right",
            "bot-left"
        ]:
            print(
                f"  {name:12s}: {points[name]}"
            )

        card_crop = perspective_crop(
            image_normalized,
            points,
            scale=scale)

        # ==========================================
        # OCR + CHECK 180°
        # ==========================================

        ocr_result = auto_orient_cascade(
            crop=card_crop,
            ocr_function=my_ocr_function,
            fast_threshold=5.0,
            ocr_threshold=5.0,
            debug=True
        )

        if ocr_result is None or ocr_result["crop"] is None:
            print("❌ Auto orientation failed")

            return {
                "image": image_normalized,
                "crop": card_crop,
                "points": points,
                "points_before_expand": points_before_expand,
                "detections": detections,
                "angle": selected_angle,
                "method": method + " | LOW OCR SCORE",
                "orientation": orientation,
                "ocr_score": ocr_score,
                "debug_points_image": debug_img,
                "debug_crop": card_crop
            }

        orientation = ocr_result["angle"]
        ocr_score = ocr_result["score"]
        card_crop = ocr_result["crop"]

        # ==========================================
        # CHECK CROP QUALITY
        # ==========================================

        good, score, reason = validate_crop_quality(card_crop)

        print(f"Crop quality: {good} | "
              f"score={score:.3f} | {reason}")

        if not good:

            return {
                "image": image_normalized,
                "crop": card_crop,
                "points": points,
                "points_before_expand": points_before_expand,
                "detections": detections,
                "angle": selected_angle,
                "method": method + " | LOW CROP QUALITY",
                "orientation": orientation,
                "ocr_score": ocr_score,
                "debug_points_image": debug_img,
                "debug_crop": card_crop
            }



        if card_crop is not None:

            print(
                "Crop size:",
                card_crop.shape[:2]
            )

        else:

            print("❌ perspective_crop returned None")


    # =====================================================
    # 9. Return
    # =====================================================

    return {
        "image": image_normalized,

        "crop": card_crop,

        "points": points,

        "points_before_expand":
            points_before_expand,

        "detections": detections,

        "angle": selected_angle,

        "method": method,

        "orientation": orientation,

        #"ocr_text": text,

        "ocr_score": ocr_score,

        "debug_points_image": debug_img,

        "debug_crop": card_crop
    }

In [ ]:
#@title Final

def process_card_folder(
    input_folder,
    output_folder,
    model_v5,
    scale=1.0
):
    """
    Xử lý toàn bộ ảnh trong folder.

    Input:
        input_folder  : folder ảnh gốc
        output_folder : folder lưu ảnh CCCD đã crop
        model_v5      : model YOLOv5
        scale         : scale perspective crop

    Return:
        results: danh sách kết quả từng ảnh
    """

    os.makedirs(
        output_folder,
        exist_ok=True
    )

    # Các định dạng ảnh
    extensions = [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.bmp",
        "*.webp"
    ]

    image_paths = []

    for ext in extensions:
        image_paths.extend(
            glob.glob(
                os.path.join(
                    input_folder,
                    ext
                )
            )
        )

    image_paths.sort()

    print(
        f"Found {len(image_paths)} images"
    )

    results = []

    for i, image_path in enumerate(
        image_paths,
        start=1
    ):

        filename = os.path.basename(
            image_path
        )

        print(
            f"\n{'='*60}"
        )

        print(
            f"[{i}/{len(image_paths)}] "
            f"{filename}"
        )

        # =============================================
        # Đọc ảnh
        # =============================================

        image = cv2.imread(
            image_path
        )

        if image is None:

            print(
                "ERROR: Cannot read image"
            )

            results.append({
                "file": filename,
                "status": "read_error"
            })

            continue

        try:

            # =========================================
            # Process CCCD
            # =========================================

            result = process_card(
                image,
                model_v5,
                crop=True,
                scale=scale
            )

            crop = result["crop"]

            # =========================================
            # Lưu output
            # =========================================

            output_path = os.path.join(
                output_folder,
                filename
            )

            success = cv2.imwrite(
                output_path,
                crop
            )

            if not success:

                print(
                    "ERROR: Cannot save output"
                )

                results.append({
                    "file": filename,
                    "status": "save_error"
                })

                continue

            print(
                f"SUCCESS -> {output_path}"
            )

            # =========================================
            # Lưu thông tin
            # =========================================

            results.append({
                "file": filename,
                "status": "success",
                "method": result["method"],
                "angle": result["angle"],
                "points": result["points"],
                "output": output_path
            })

        except Exception as e:

            print(
                f"ERROR: {type(e).__name__}: {e}"
            )

            results.append({
                "file": filename,
                "status": "error",
                "error": str(e)
            })

    # =================================================
    # Summary
    # =================================================

    success_count = sum(
        r["status"] == "success"
        for r in results
    )

    error_count = len(results) - success_count

    print(
        f"\n{'='*60}"
    )

    print("DONE")

    print(
        f"Success: {success_count}"
    )

    print(
        f"Error:   {error_count}"
    )

    return results

results = process_card_folder(
    input_folder="/content/input",
    output_folder="/content/output",
    model_v5=model_v5,
    scale=1.0
)

!zip -r /content/cccd_processed.zip /content/output
!rm -rf /content/output

from google.colab import files
files.download("/content/cccd_processed.zip")

#clear_output()

In [ ]:
#@title Your test here

upload_folder = '/content/user_upload'

if not os.path.exists(upload_folder):
    os.mkdir(upload_folder)
target_person=[]
uploaded = files.upload()
for filename in uploaded.keys():
  dst_path = os.path.join(upload_folder, filename)
  print(f'move {filename} to {dst_path}')
  shutil.move(filename, dst_path)
  target_person.append(dst_path)

clear_output()
target_person[-1]

img_path = '/content/user_upload/' + filename
image = cv2.imread(img_path)

cv2_imshow(process_card(image, model_v5, crop=True, scale=1.0)['crop'])

In [ ]:
#cv2_imshow(perspective_crop(image, complete_card_points(image, get_detections(model_v5, image))[0], scale=1.0))

cv2_imshow(process_card(image, model_v5, crop=True, scale=1.0)['crop'])